# Freeze final candidate protocol
Project: DDoS_Prevention_and_Mitigation. Kaggle CPU; internet enabled. Source: https://github.com/saltypal/DDoS_Prevention_and_Mitigation.git
Rejects resource-pilot runs. Requires reviewed matched training artifacts; final tests remain unopened.
Dependency specification: pyproject.toml in the exact source commit. Bootstrap prints the actual
Python, Java and installed dependencies. No local environment transfers to Kaggle.
Attach seyhed/nf-ton-iot-v3 and seyhed/nf-bot-iot-v3. Official LAN v3.0 is downloaded remotely.
Modules run stage by stage and write incremental logs, Parquet, figures and provenance under
/kaggle/working/experiments/public_dataset_la/. Saved-version persistence/readback must be verified.
Bootstrap output is available in the Kaggle kernel log; subsequent cells are also captured incrementally.


In [ ]:
import sys, subprocess, shutil, uuid, time
from pathlib import Path
print('START repository bootstrap', flush=True)
started = time.monotonic()
assert Path('/kaggle/working').exists(), 'Execute scientific notebooks on Kaggle'
REPOSITORY = 'https://github.com/saltypal/DDoS_Prevention_and_Mitigation.git'
REVISION = None  # None resolves latest main ONCE. Set a recorded SHA to reproduce a run.
checkout = Path('/kaggle/temp') / ('ddos-source-' + uuid.uuid4().hex)
subprocess.run(['git','clone','--depth','1',REPOSITORY,str(checkout)],check=True,timeout=180)
if REVISION:
    subprocess.run(['git','fetch','--depth','1','origin',REVISION],cwd=checkout,check=True,timeout=180)
    subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=checkout,check=True)
SOURCE_COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],cwd=checkout,text=True).strip()
print('PINNED SOURCE',SOURCE_COMMIT,checkout,flush=True)
if not shutil.which('java'):
    subprocess.run(['apt-get','update','-qq'],check=True,timeout=300)
    subprocess.run(['apt-get','install','-y','openjdk-17-jre-headless'],check=True,timeout=600)
subprocess.run(['java','-version'],check=True)
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','-c',str(checkout/'experiments/public_dataset_la/constraints-kaggle.txt'),'-e',str(checkout)+'[spark,boost,notebooks]'],check=True,timeout=900)
sys.path.insert(0,str(checkout))
import os
os.chdir(checkout)
print('END repository bootstrap',time.monotonic()-started,'seconds',flush=True)
from sdn_ddos.runtime.notebook import install_capture
capture_root=Path('/kaggle/working/experiments/public_dataset_la/cell_capture') / uuid.uuid4().hex
install_capture(capture_root)


In [ ]:
import json
from sdn_ddos.cli import execute
print('START configuration', flush=True)
config = json.loads((checkout / 'experiments/public_dataset_la/configs/freeze.json').read_text())
print(json.dumps(config,indent=2), 'SOURCE', SOURCE_COMMIT, flush=True)
print('END configuration', flush=True)


In [ ]:
print('START pipeline', flush=True)
output = execute(config)
print('END pipeline; artifacts:',output,flush=True)

## Interpretation and limitations
Inspect reports and source-specific class counts before accepting artifacts. Prefix smoke outputs
are not training data or performance evidence. Do not claim universal detection or distributed
Spark scaling from a local Kaggle executor. Final tests must remain outside model selection.
